In [ ]:
# coding=utf-8
import os
import json
import subprocess as sp
from io import StringIO
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from jinja2 import Environment, FileSystemLoader

# Airflow & Pharos Imports (commented out for notebook usage)
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.utils.email import send_email
# import pendulum
# from com.workday.pharos.persistence.pharos_persistence import PharosPersistence

In [ ]:
# --- Configuration & Constants ---
DAG_HOME = os.path.dirname(os.path.abspath("__file__"))

email_list = [
    "huzefa.saifee@workday.com",
    "m6a0l2y5u3c9i6f3@workday.enterprise.slack.com",
]

In [ ]:
def render_sql(filename, **kwargs):
    """Loads and renders a Jinja2 template SQL file dynamically."""
    env = Environment(loader=FileSystemLoader(DAG_HOME))
    template = env.get_template(filename)
    return template.render(**kwargs)

def run_cli_fetch_json(cmd):
    """Executes a pharos CLI command, parses the stdout as JSON, and returns the result.data (CSV)."""
    result = sp.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}\nCMD: {cmd}\nSTDERR: {result.stderr}")
    
    raw_output = result.stdout.strip()
    try:
        parsed = json.loads(raw_output)
        return parsed["result"]["data"]
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Failed to parse JSON output. Command: {cmd}\nOutput: {raw_output[:300]}") from e

def fetch_data(file_name, str_month_to_query_from):
    """Renders the SQL, runs it via pharos cli, and loads the CSV into a pandas DataFrame."""
    swh_query = render_sql(f"{file_name}.sql", oldest_month_value=str_month_to_query_from)
    
    # Run the query and extract the data
    cmd = f'pharos sql run --sql "{swh_query}"'
    csv_data = run_cli_fetch_json(cmd)
    
    # Load to DataFrame
    df = pd.read_csv(StringIO(csv_data))
    return df

In [ ]:
# Determine the query date (24 months ago)
date_24_month_ago = datetime.today().replace(day=1) - relativedelta(months=24)
month_to_query_from = date_24_month_ago.replace(hour=0, minute=0, second=0, microsecond=0)
str_month_to_query_from = month_to_query_from.strftime("%Y-%m-%d")

data_list = []
sql_files = [
    "scopes_metrics",
    "scopes_input_type_metrics",
    "scopes_selection_type_metrics",
    "scopes_validation_usages_metrics",
    "scopes_materialization_metrics",
]

for sql_file in sql_files:
    print(f"Processing: {sql_file}")
    df = fetch_data(sql_file, str_month_to_query_from)
    print(f"Fetched {len(df)} rows for {sql_file}.")
    data_list.append(dict({"name": sql_file, "df": df}))

In [ ]:
for dt in data_list:
    dt["df"].to_csv(os.path.join(DAG_HOME, f"{dt['name']}.csv"), index=False)